# Архитектор

## Настройка

In [3]:
!sudo apt-get update
!sudo apt-get install -y pciutils zstd

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
!pip install langgraph langchain-ollama pydantic

In [8]:
import subprocess
import time

process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print("Ollama сервер запускается...")
time.sleep(8)
print("✅ Ollama готов!")

Ollama сервер запускается...
✅ Ollama готов!


In [29]:
!ollama pull qwen2.5:7b-instruct

In [30]:
!ollama list

NAME                   ID              SIZE      MODIFIED      
qwen2.5:7b-instruct    845dbda0ea48    4.7 GB    2 minutes ago    


In [16]:
import json
import re
import sys
import functools
from pathlib import Path
from typing import TypedDict, Optional

from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama

In [17]:
PROMPTS_DIR = Path("prompts")

## Утилиты

In [33]:
def load_prompt(name: str) -> str:
    path = PROMPTS_DIR / f"{name}.txt"
    if not path.exists():
        raise FileNotFoundError(f"Промпт не найден: {path}")
    return path.read_text(encoding="utf-8")


def fill_prompt(name: str, **kwargs) -> str:
    """Загружает промпт и подставляет переменные через str.replace.
    В отличие от .format() не ломается на фигурных скобках внутри JSON-примеров."""
    text = load_prompt(name)
    for key, value in kwargs.items():
        text = text.replace("{" + key + "}", value)
    return text


def safe_json_parse(text: str) -> dict:
    """Надёжное извлечение JSON из ответа LLM."""
    text = re.sub(r"```(?:json)?\s*", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    raise ValueError(f"Не удалось распарсить JSON:\n{text[:500]}")


def print_section(title: str, data: dict | list) -> None:
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(json.dumps(data, indent=2, ensure_ascii=False))


def ask_user(prompt: str, valid_options: list[str]) -> str:
    while True:
        print(f"\n{prompt}  [{' / '.join(valid_options)}]")
        answer = input("> ").strip().lower()
        if answer in valid_options:
            return answer
        print(f"  Введите один из: {valid_options}")

## Стейт

In [20]:
class ArchitectState(TypedDict):
    raw_document:         str
    semantic_map:         Optional[dict]
    clarifying_questions: Optional[dict]
    epics:                Optional[list[dict]]
    stories:              Optional[list[dict]]
    all_stories:          list[dict]
    tasks:                Optional[list[dict]]
    all_tasks:            list[dict]
    qa_result:            Optional[dict]
    qa_retries:           int
    current_epic_index:   int
    user_feedback:        Optional[str]

## Nodes

In [34]:
MAX_QA_RETRIES = 2


def analyze_document(state: ArchitectState, llm) -> dict:
    print("\n🔍 Анализирую документ...")
    prompt = load_prompt("analyze_document")
    response = llm.invoke(prompt + "\n\n" + state["raw_document"])
    semantic_map = safe_json_parse(response.content)
    print_section("📋 Семантическая карта", semantic_map)
    return {"semantic_map": semantic_map}


def handle_clarifying_questions(state: ArchitectState, llm) -> dict:
    open_questions = state["semantic_map"].get("open_questions", [])
    if not open_questions:
        print("\n✅ Открытых вопросов нет, продолжаем...")
        return {"clarifying_questions": None}

    prompt = fill_prompt("clarifying_questions",
        open_questions=json.dumps(open_questions, ensure_ascii=False),
        semantic_map=json.dumps(state["semantic_map"], ensure_ascii=False),
    )
    questions = safe_json_parse(llm.invoke(prompt).content)
    print_section("❓ Уточняющие вопросы", questions)

    all_q = [q for g in questions.get("question_groups", []) for q in g.get("questions", [])]
    high  = [q for q in all_q if q.get("criticality") == "high"]

    answers = {}
    if high:
        print("\n⚠️  Есть критичные вопросы — без ответов декомпозиция будет неточной.")
        if ask_user("Ответить сейчас?", ["y", "n"]) == "y":
            for q in high:
                print(f"\n[{q['id']}] {q['question']}")
                print(f"    Почему важно: {q['why_important']}")
                ans = input("    Ваш ответ (Enter — пропустить): ").strip()
                if ans:
                    answers[q["id"]] = ans

    semantic_map = state["semantic_map"].copy()
    if answers:
        semantic_map["clarifications"] = answers
    return {"clarifying_questions": questions, "semantic_map": semantic_map}


def generate_epics(state: ArchitectState, llm) -> dict:
    print("\n⚙️  Генерирую эпики...")
    context  = json.dumps(state["semantic_map"], ensure_ascii=False)
    feedback = state.get("user_feedback")
    suffix   = f"\n\nФидбек от заказчика: {feedback}" if feedback else ""
    response = llm.invoke(load_prompt("generate_epics") + "\n\n" + context + suffix)
    parsed   = safe_json_parse(response.content)
    print_section("🗂️  Эпики", parsed["epics"])
    return {"epics": parsed["epics"], "user_feedback": None}


def qa_epics(state: ArchitectState, llm) -> dict:
    print("\n🔍 QA эпиков...")
    prompt = fill_prompt("qa_epics",
        semantic_map=json.dumps(state["semantic_map"], ensure_ascii=False),
        epics=json.dumps(state["epics"], ensure_ascii=False),
    )
    parsed = safe_json_parse(llm.invoke(prompt).content)
    print_section("🔍 Результат QA", parsed)
    return {"qa_result": parsed}


def decide_after_qa_epics(state: ArchitectState) -> str:
    qa      = state["qa_result"]
    retries = state.get("qa_retries", 0)

    if qa["status"] == "fail" and retries < MAX_QA_RETRIES:
        print(f"\n⚠️  QA не прошёл (попытка {retries + 1}/{MAX_QA_RETRIES}):")
        for issue in qa.get("issues", []):
            print(f"   • [{issue['type']}] {issue['description']}")
        print("\n🔄 Перегенерирую автоматически...")
        return "regenerate_epics"

    if qa["status"] == "fail":
        print("\n⚠️  Лимит QA-попыток исчерпан.")

    print("\n  y — утвердить и перейти к User Stories")
    print("  e — переделать (опишите что не так)")
    print("  q — выйти")
    choice = ask_user("Ваш выбор:", ["y", "e", "q"])

    if choice == "y":
        return "generate_stories"
    elif choice == "e":
        feedback = input("Что нужно изменить: ").strip()
        state["semantic_map"]["epic_feedback"] = feedback
        return "regenerate_epics"
    else:
        return "end"


def increment_qa_retries(state: ArchitectState) -> dict:
    return {"qa_retries": state.get("qa_retries", 0) + 1}


def generate_stories(state: ArchitectState, llm) -> dict:
    idx   = state.get("current_epic_index", 0)
    epics = state["epics"]

    if idx >= len(epics):
        return {}

    epic = epics[idx]
    print(f"\n📖 User Stories — эпик [{idx + 1}/{len(epics)}]: {epic['title']}")

    base_prompt = fill_prompt("generate_stories",
        epic_id=epic["id"],
        epic=json.dumps(epic, ensure_ascii=False),
        semantic_map=json.dumps(state["semantic_map"], ensure_ascii=False),
    )
    prompt = base_prompt

    while True:
        stories = safe_json_parse(llm.invoke(prompt).content)["stories"]
        print_section(f"📝 User Stories — {epic['title']}", stories)

        print("\n  y    — утвердить")
        print("  r    — переделать")
        print("  skip — пропустить эпик")
        choice = ask_user("Ваш выбор:", ["y", "r", "skip"])

        if choice == "y":
            return {
                "stories":     stories,
                "all_stories": state.get("all_stories", []) + stories,
                "current_epic_index": idx + 1,
            }
        elif choice == "skip":
            return {"stories": [], "current_epic_index": idx + 1}
        else:
            feedback = input("Что нужно исправить: ").strip()
            prompt = base_prompt + f"\n\nФидбек: {feedback}"


def decide_after_stories(state: ArchitectState) -> str:
    idx     = state.get("current_epic_index", 0)
    stories = state.get("stories", [])
    if not stories:
        return "finalize" if idx >= len(state["epics"]) else "generate_stories"
    return "finalize" if idx >= len(state["epics"]) else "generate_tasks"


def generate_tasks(state: ArchitectState, llm) -> dict:
    idx  = state.get("current_epic_index", 1)
    epic = state["epics"][idx - 1]
    print(f"\n⚙️  Задачи для эпика: {epic['title']}")

    prompt = fill_prompt("generate_tasks",
        story_id=epic["id"],
        stories=json.dumps(state.get("stories", []), ensure_ascii=False),
        epic=json.dumps(epic, ensure_ascii=False),
    )
    tasks = safe_json_parse(llm.invoke(prompt).content)["tasks"]
    print_section(f"✅ Задачи — {epic['title']}", tasks)

    return {
        "tasks":     tasks,
        "all_tasks": state.get("all_tasks", []) + tasks,
    }


def decide_next_epic(state: ArchitectState) -> str:
    idx = state.get("current_epic_index", 0)
    return "generate_stories" if idx < len(state["epics"]) else "finalize"


def finalize_and_export(state: ArchitectState) -> dict:
    epics       = state.get("epics", [])
    all_stories = state.get("all_stories", [])
    all_tasks   = state.get("all_tasks", [])

    stories_by_epic  = {}
    for s in all_stories:
        stories_by_epic.setdefault(s.get("epic_id", "?"), []).append(s)

    tasks_by_story = {}
    for t in all_tasks:
        tasks_by_story.setdefault(t.get("story_id", "?"), []).append(t)

    print("\n" + "=" * 60)
    print("🎉  ИТОГОВАЯ ДЕКОМПОЗИЦИЯ")
    print("=" * 60)

    for epic in epics:
        print(f"\n📦 [{epic['id']}] {epic['title']}")
        print(f"   {epic['description']}")
        print(f"   Ценность: {epic['business_value']}")

        for story in stories_by_epic.get(epic["id"], []):
            print(f"\n   📝 [{story['id']}] {story['title']}")
            print(f"       Как {story['as_a']}, я хочу {story['i_want']},")
            print(f"       чтобы {story['so_that']}")
            for ac in story.get("acceptance_criteria", []):
                print(f"         ✓ {ac}")

            for task in tasks_by_story.get(story["id"], []):
                print(f"\n       ⚙️  [{task['id']}] [{task['type'].upper()}] {task['title']}")
                print(f"           {task['description']}")
                print(f"           Оценка: {task['estimate_hours']}ч  |  DoD: {task['definition_of_done']}")

    print("\n" + "=" * 60)
    print(f"Итого: {len(epics)} эпиков | {len(all_stories)} stories | {len(all_tasks)} задач")
    print("=" * 60)
    return {}

## Граф

In [35]:
def build_graph(llm):
    def node(fn):
        return functools.partial(fn, llm=llm)

    wf = StateGraph(ArchitectState)

    wf.add_node("analyze",          node(analyze_document))
    wf.add_node("clarify",          node(handle_clarifying_questions))
    wf.add_node("generate_epics",   node(generate_epics))
    wf.add_node("qa_epics",         node(qa_epics))
    wf.add_node("inc_retries",      increment_qa_retries)
    wf.add_node("generate_stories", node(generate_stories))
    wf.add_node("generate_tasks",   node(generate_tasks))
    wf.add_node("finalize",         finalize_and_export)

    wf.set_entry_point("analyze")

    wf.add_edge("analyze",        "clarify")
    wf.add_edge("clarify",        "generate_epics")
    wf.add_edge("generate_epics", "qa_epics")

    wf.add_conditional_edges("qa_epics", decide_after_qa_epics, {
        "regenerate_epics": "inc_retries",
        "generate_stories": "generate_stories",
        "end":              END,
    })
    wf.add_edge("inc_retries", "generate_epics")

    wf.add_conditional_edges("generate_stories", decide_after_stories, {
        "generate_tasks":   "generate_tasks",
        "generate_stories": "generate_stories",
        "finalize":         "finalize",
    })

    wf.add_conditional_edges("generate_tasks", decide_next_epic, {
        "generate_stories": "generate_stories",
        "finalize":         "finalize",
    })

    wf.add_edge("finalize", END)
    return wf.compile()

## Load

In [36]:
def run(document_path: str, model: str = "qwen2.5:7b-instruct") -> dict:
    with open(document_path, encoding="utf-8") as f:
        document = f.read()

    print(f"\n🚀 IT Project Architect")
    print(f"   Документ : {document_path} ({len(document)} символов)")
    print(f"   Модель   : {model}\n")

    llm   = ChatOllama(model=model, temperature=0.2, format="json", num_ctx=8192, repeat_penalty=1.1)
    graph = build_graph(llm)

    return graph.invoke({
        "raw_document":         document,
        "semantic_map":         None,
        "clarifying_questions": None,
        "epics":                None,
        "stories":              None,
        "all_stories":          [],
        "tasks":                None,
        "all_tasks":            [],
        "qa_result":            None,
        "qa_retries":           0,
        "current_epic_index":   0,
        "user_feedback":        None,
    })

## Запуск

In [37]:
run("input.txt")


🚀 IT Project Architect
   Документ : input.txt (2168 символов)
   Модель   : qwen2.5:7b-instruct


🔍 Анализирую документ...

  📋 Семантическая карта
{
  "domains": [
    {
      "name": "Корпоративное обучение",
      "description": "Платформа для создания, назначения и отслеживания учебных программ для корпоративных клиентов."
    },
    {
      "name": "Аутентификация и авторизация",
      "description": "Система управления доступом пользователей к платформе с использованием JWT и SSO."
    }
  ],
  "entities": [
    {
      "name": "Корпорация",
      "description": "Клиент, который использует платформу для обучения своих сотрудников.",
      "domain": "Корпоративное обучение"
    },
    {
      "name": "Пользователь",
      "description": "Любой участник системы с определенной ролью (администратор компании, администратор платформы или сотрудник).",
      "domain": "Корпоративное обучение"
    },
    {
      "name": "Курс",
      "description": "Учебная программа, состоящая из моду

KeyboardInterrupt: Interrupted by user